# Notebook 08: Ray Data

## Why This Matters

For large-scale pre-training (GPT-3 scale, 300B+ tokens), the data pipeline is often the bottleneck. PyTorch DataLoader works well for fine-tuning but struggles when the dataset does not fit in RAM, requires heavy preprocessing, or needs to be shuffled at petabyte scale. Ray Data provides a lazy, streaming, distributed data processing API that can ingest from S3/GCS, run parallel transformations, and feed Ray Train without materializing the entire dataset in memory. This is the data layer for pre-training and large-scale ML.


In [ ]:
%pip install -q 'ray[data]' numpy pandas pyarrow torch

In [ ]:
import ray
import numpy as np
import pandas as pd
import torch
import time
import tempfile, os

if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, ignore_reinit_error=True)
print(f"Ray version: {ray.__version__}")
print("Ready.")

## 1. Dataset vs DataLoader

| Aspect | PyTorch DataLoader | Ray Data |
|--------|-------------------|---------|
| Scale | Single machine | Multi-machine |
| Memory | All data loaded | Streaming, lazy |
| Source | Files / in-memory | S3, GCS, local, databases |
| Transforms | Python functions | Parallel (map_batches) |
| Shuffle | In-memory | Streaming local shuffle |
| Integration | PyTorch native | Ray Train, TensorFlow |

**When to use Ray Data over DataLoader**:
- Dataset does not fit in RAM
- Transforms are CPU-intensive (tokenization, image augmentation at scale)
- Data is on object storage (S3, GCS)
- You want distributed preprocessing that runs in parallel with training


In [ ]:
# Ray Data basics: creating datasets

# From in-memory list
ds_list = ray.data.from_items([{'x': i, 'y': i*2} for i in range(100)])
print('From list:', ds_list)

# From pandas
df = pd.DataFrame({'text': [f'sample {i}' * 10 for i in range(1000)],
                   'label': np.random.randint(0, 5, 1000)})
ds_pandas = ray.data.from_pandas(df)
print('From pandas:', ds_pandas)

# From numpy
X = np.random.randn(1000, 128).astype(np.float32)
Y = np.random.randint(0, 10, 1000)
ds_numpy = ray.data.from_numpy(X)  # features
print('From numpy:', ds_numpy)

# Create a parquet dataset (simulating object storage)
with tempfile.TemporaryDirectory() as tmpdir:
    # Write parquet files
    for i in range(4):
        chunk = df.iloc[i*250:(i+1)*250]
        chunk.to_parquet(f'{tmpdir}/shard_{i:04d}.parquet')
    
    # Read from parquet (lazy -- doesn't load everything)
    ds_parquet = ray.data.read_parquet(tmpdir)
    print('From parquet:', ds_parquet)
    print('  Schema:', ds_parquet.schema())
    print('  Count:', ds_parquet.count())

## 2. Transformations: map, filter, map_batches

Ray Data transformations are **lazy** -- they build a computation graph but do not execute until you iterate. This allows Ray to optimize the execution plan.

Key transforms:
- `map(fn)`: apply fn to each row independently
- `filter(fn)`: keep rows where fn returns True
- `map_batches(fn)`: apply fn to batches (faster, enables vectorized operations)
- `flat_map(fn)`: fn returns a list, results are flattened

For neural network preprocessing, always use `map_batches` with `batch_format='numpy'` -- this enables vectorized operations and GPU acceleration.


In [ ]:
# Ray Data transformations

# Create a dataset
np.random.seed(42)
texts = [{'text': f'token_{i} ' * np.random.randint(10, 50), 'label': i % 5}
         for i in range(500)]
ds = ray.data.from_items(texts)
print('Original dataset:', ds)

# map: per-row transformation
ds_mapped = ds.map(lambda row: {'text': row['text'].upper(),
                                  'label': row['label'],
                                  'length': len(row['text'].split())})

# filter: keep only rows with sufficient length
ds_filtered = ds_mapped.filter(lambda row: row['length'] >= 20)

# map_batches: batch transformation (better for vectorized ops)
def tokenize_batch(batch):
    # Simulate tokenization (in practice: transformers tokenizer)
    texts = batch['text']
    lengths = np.array([len(t.split()) for t in texts])
    return {'text': texts, 'label': batch['label'], 'token_count': lengths}

ds_tokenized = ds_filtered.map_batches(tokenize_batch, batch_format='pandas')

# Count (forces execution)
count = ds_tokenized.count()
print(f'After filter: {count} rows (from 500)')

# Show schema
print('Schema:', ds_tokenized.schema())

# Take first few rows
sample = ds_tokenized.take(3)
for row in sample:
    print(f'  token_count={row["token_count"]}, label={row["label"]}')

## 3. Streaming and Shuffling

Ray Data's streaming iterator enables processing datasets larger than RAM. The key API: `iter_torch_batches()` yields batches as PyTorch tensors, ready for training.

**Shuffle tradeoff**:
- **Global shuffle**: perfect randomness, requires full dataset pass -- impractical for streaming
- **Local shuffle**: buffer `n` examples, shuffle within buffer -- good approximation, streaming-friendly
- **No shuffle**: fast but data order affects convergence

For pre-training, local shuffle with buffer_size=10000+ is standard practice.


In [ ]:
# Streaming iterator for training

# Create a dataset with features and labels
N = 2000
features = np.random.randn(N, 64).astype(np.float32)
labels = np.random.randint(0, 10, N).astype(np.int64)

ds = ray.data.from_numpy({'X': features, 'y': labels})

# Shuffle with local buffer
ds_shuffled = ds.random_shuffle()  # global shuffle (loads all data)

print('Iterating with iter_torch_batches:')
batch_sizes = []
t0 = time.perf_counter()

for i, batch in enumerate(ds.iter_torch_batches(batch_size=64, dtypes={'X': torch.float32, 'y': torch.long})):
    batch_sizes.append(batch['X'].shape[0])
    if i >= 5:
        break

t_iter = time.perf_counter() - t0
print(f'First 6 batch sizes: {batch_sizes}')
print(f'Iteration setup time: {t_iter:.3f}s')

# Integration with training loop
print('\nIntegration pattern with training:')
train_code = '''
# In train_loop_per_worker:
ds = train.get_dataset_shard("train")  # get this worker's data shard

for epoch in range(n_epochs):
    for batch in ds.iter_torch_batches(batch_size=64):
        x = batch["X"].to(device)
        y = batch["y"].to(device)
        loss = model(x, y)
        ...
'''
print(train_code)

## Summary

### Key Concepts

| Concept | API | Notes |
|---------|-----|-------|
| Create dataset | `ray.data.from_items/pandas/numpy/parquet` | Lazy by default |
| Transform per-row | `ds.map(fn)` | For row-level transforms |
| Transform batches | `ds.map_batches(fn, batch_format='numpy')` | Vectorized, faster |
| Filter | `ds.filter(fn)` | Keep matching rows |
| Shuffle | `ds.random_shuffle()` or `local_shuffle_buffer_size` | Global vs local tradeoff |
| Training iterator | `ds.iter_torch_batches(batch_size=N)` | Yields PyTorch tensors |

**Key insight**: For datasets > 100GB, use `ray.data.read_parquet(s3://...)` + `map_batches` + `iter_torch_batches`. This never loads the full dataset into RAM and parallelizes preprocessing across all CPUs automatically.

**Next**: `09_ray_serve.ipynb` -- deploying models with Ray Serve.
